## Todo list:

1. do modulo with 2^32 in arithmetic operations

In [164]:
sample_strings = [
    "00000000001100010000000010110011",
    "01000000100000101000001000110011",
    "00000000100101000000001110110011",
    "01000000010100110000010010110011",
    "00000000010000010010001010000011",
    "11111111100000101010001100000011",
    "00000000110001001010010000000011",
    "00000000010100010010010000100011",
    "11111110011000101010100000100011",
    "00000000101001001010011000100011"
]

In [165]:
register_file = ["0"*32] * 32

In [166]:
memory = ["0"*8] * 4096 # 4096 bytes of memory, each byte is 8 bits

## Buffer Registers:

In [167]:
# Type definitions for pipeline Buffers

from typing import TypedDict

class RF_EX_Buffer_Class(TypedDict):
    A: int
    B: int
    rs1: str
    rs2: str
    rd: str
    funct7: str
    isAdd: bool
    isSub: bool
    isLd: bool
    isSt: bool
    doWB: bool

class EX_MEM_Buffer_Class(TypedDict):
    alu_result: int
    rs2_value: int
    rd: str
    isLd: bool
    isSt: bool
    doWB: bool

class MEM_WB_Buffer_Class(TypedDict):
    writeback_value: int
    rs2_value: int
    rd: str
    isLd: bool
    doWB: bool


In [168]:
# Declaring pipeline buffer objects

RF_EX_Buffer: RF_EX_Buffer_Class = {
    "A": 0,
    "B": 0,
    "rs1": "0"*5,
    "rs2": "0"*5,
    "rd": "0"*5,
    "funct7": "0"*7,
    "isAdd": False,
    "isSub": False,
    "isLd": False,
    "isSt": False,
    "doWB": False
}

EX_MEM_Buffer: EX_MEM_Buffer_Class = {
    "alu_result": 0,
    "rs2_value": 0,
    "rd": "0"*5,
    "isLd": False,
    "isSt": False,
    "doWB": False
}

MEM_WB_Buffer: MEM_WB_Buffer_Class = {
    "writeback_value": 0,
    "rs2_value": 0,
    "rd": "0"*5,
    "isLd": False,
    "doWB": False
}

## Register Fetch / Instruction decode stage

In [169]:
# Core Register Fetch / Instruction decode

def get_opcode(instruction: str) -> str:
    return instruction[25:]

def get_funct3(instruction: str) -> str:
    return instruction[17:20]

def get_funct7(instruction: str) -> str:
    return instruction[0:7]

def get_rs1(instruction: str) -> int:
    register_index_string = instruction[12:17]
    register_value_binary = register_file[int(register_index_string, 2)]
    register_value = int(register_value_binary, 2)
    return register_value

def get_rs2(instruction: str) -> int:
    register_index_string = instruction[7:12]
    register_value_binary = register_file[int(register_index_string, 2)]
    register_value = int(register_value_binary, 2)
    return register_value

def get_rd(instruction: str):
    return instruction[20:25]

In [170]:
# Control Unit

def control_unit(instruction):
    assert len(instruction) == 32, "Instruction must be 32 bits long"
    
    opcode = get_opcode(instruction)
    funct3 = get_funct3(instruction)
    funct7 = get_funct7(instruction)

    isAdd = (opcode == "0110011" and funct3 == "000" and funct7 == "0000000")
    isSub = (opcode == "0110011" and funct3 == "000" and funct7 == "0100000")
    
    isLd = (opcode == "0000011")
    isSt = (opcode == "0100011")

    doWB = isAdd or isSub or isLd

    return {
        "isAdd": isAdd,
        "isSub": isSub,
        "isLd": isLd,
        "isSt": isSt,
        "doWB": doWB
    }

In [171]:
# main function of instruction decode stage

def instruction_decode(instruction):
    RF_EX_Buffer["A"] = get_rs1(instruction)
    RF_EX_Buffer["B"] = get_rs2(instruction)
    RF_EX_Buffer["rd"] = get_rd(instruction)

    control_output = control_unit(instruction)
    RF_EX_Buffer["isAdd"] = control_output["isAdd"]
    RF_EX_Buffer["isSub"] = control_output["isSub"]
    RF_EX_Buffer["isLd"] = control_output["isLd"]
    RF_EX_Buffer["isSt"] = control_output["isSt"]
    RF_EX_Buffer["doWB"] = control_output["doWB"]

In [172]:
# Branch unit implementation

## Execute Stage

In [173]:
# A, B resolution for ALU

def input_resolve():
    A = RF_EX_Buffer["A"]

    # these cases are for immediate values in load and store instructions
    if RF_EX_Buffer["isLd"]:
        B = int(RF_EX_Buffer["funct7"] + RF_EX_Buffer["rs2"], 2)
    elif RF_EX_Buffer["isSt"]:
        B = int(RF_EX_Buffer["funct7"] + RF_EX_Buffer["rd"], 2)
    else:
        B = RF_EX_Buffer["B"]

    return A, B

In [174]:
# Execute stage

def Adder(A, B, isAdd, isSub, isLd, isSt) -> int:
    assert sum([isAdd, isSub, isLd, isSt]) == 1, "Exactly one of isAdd, isSub, isLd, isSt must be True"

    if isAdd or isLd or isSt:
        return A + B
    else:   # isSub
        return A - B

def Multiplier(A, B, isMul):
    return 0

def Divider(A, B, isDiv, isMod):
    return 0

def ShiftUnit(A, B, isLsl, isLsr, isAsr):
    return 0

def LogicalUnit(A, B, isAnd, isOr, isNot):
    return 0

def isMov():
    return 0


In [175]:
def execute():
    A, B = input_resolve()
    
    EX_MEM_Buffer["rd"] = RF_EX_Buffer["rd"]

    EX_MEM_Buffer["isLd"] = RF_EX_Buffer["isLd"]
    EX_MEM_Buffer["isSt"] = RF_EX_Buffer["isSt"]

    EX_MEM_Buffer["alu_result"] = Adder(A, B, RF_EX_Buffer["isAdd"], RF_EX_Buffer["isSub"], RF_EX_Buffer["isLd"], RF_EX_Buffer["isSt"])

    EX_MEM_Buffer["doWB"] = RF_EX_Buffer["doWB"]

    # for store instruction, we need to pass the value of rs2 to memory access stage, so storing it in EX_MEM_Buffer
    EX_MEM_Buffer["rs2_value"] = RF_EX_Buffer["B"]

## Memory Stage

##### Note: Machine is Big Endian

In [176]:
def memory_access():
    if EX_MEM_Buffer["isLd"]:
        return_value = 0

        return_value += int(memory[EX_MEM_Buffer["alu_result"]], 2) << 24
        return_value += int(memory[EX_MEM_Buffer["alu_result"] + 1], 2) << 16
        return_value += int(memory[EX_MEM_Buffer["alu_result"] + 2], 2) << 8
        return_value += int(memory[EX_MEM_Buffer["alu_result"] + 3], 2)

        MEM_WB_Buffer["writeback_value"] = return_value

    elif EX_MEM_Buffer["isSt"]:
        memory[EX_MEM_Buffer["alu_result"]] = format(EX_MEM_Buffer["rs2_value"] >> 24 & 0xFF, "08b")
        memory[EX_MEM_Buffer["alu_result"] + 1] = format((EX_MEM_Buffer["rs2_value"] >> 16) & 0xFF, "08b")
        memory[EX_MEM_Buffer["alu_result"] + 2] = format((EX_MEM_Buffer["rs2_value"] >> 8) & 0xFF, "08b")
        memory[EX_MEM_Buffer["alu_result"] + 3] = format((EX_MEM_Buffer["rs2_value"]) & 0xFF, "08b")

    else:
        MEM_WB_Buffer["writeback_value"] = EX_MEM_Buffer["alu_result"]

    MEM_WB_Buffer["doWB"] = EX_MEM_Buffer["doWB"]
    MEM_WB_Buffer["rd"] = EX_MEM_Buffer["rd"]
    MEM_WB_Buffer["isLd"] = EX_MEM_Buffer["isLd"]
    MEM_WB_Buffer["rs2_value"] = EX_MEM_Buffer["rs2_value"]

## Register Write Back Stage

In [177]:
def write_back():
    if MEM_WB_Buffer["doWB"]:
        # convert writeback value to binary string and store in register file
        register_file[int(MEM_WB_Buffer["rd"], 2)] = format(MEM_WB_Buffer["writeback_value"], '032b')

        assert len(register_file[int(MEM_WB_Buffer["rd"], 2)]) == 32, "Register value must be 32 bits long"


## Testing

In [ ]:
# register_file[3] = "00000000000000000000000000000101" # x3 = 5
# register_file[2] = "00000000000000000000000000001010" # x2 = 10

# test_instruction = "00000000001100010000000010110011" # add x1, x2, x3

memory[2048] = "01010101"
memory[2049] = "01010101"
memory[2050] = "01010101"
memory[2051] = "01010101"

register_file[2] = "00000000000000000000100000000000" # x2 = 2048
register_file[4] = "00000000000000001111111111111111"

test_instruction = "00000000010000010010000000100011" # st x4, 0(x2) -> store value of x4 to memory[2048]
# test_instruction = "00000000000000010000000110000011" # ld x3, 0(x2) -> load value at memory[2048] to x3

instruction_decode(test_instruction)
print("RF_EX_Buffer after instruction decode:")
print(RF_EX_Buffer)
execute()
print("EX_MEM_Buffer after execution:")
print(EX_MEM_Buffer)
memory_access()
print("MEM_WB_Buffer after memory access:")
print(MEM_WB_Buffer)
write_back()

print("Register file after execution:")
for i in range(len(register_file)):
    print(f"x{i}: {register_file[i]}")

print("Memory after execution:")
for i in range(2048, 2052):
    print(f"memory[{i}]: {memory[i]}")

RF_EX_Buffer after instruction decode:
{'A': 2048, 'B': 65535, 'rs1': '00000', 'rs2': '00000', 'rd': '00000', 'funct7': '0000000', 'isAdd': False, 'isSub': False, 'isLd': False, 'isSt': True, 'doWB': False}
EX_MEM_Buffer after execution:
{'alu_result': 2048, 'rs2_value': 65535, 'rd': '00000', 'isLd': False, 'isSt': True, 'doWB': False}
MEM_WB_Buffer after memory access:
{'writeback_value': 0, 'rs2_value': 65535, 'rd': '00000', 'isLd': False, 'doWB': False}
Register file after execution:
x0: 00000000000000000000000000000000
x1: 00000000000000000000000000000000
x2: 00000000000000000000100000000000
x3: 00000000000000000000000000000000
x4: 00000000000000001111111111111111
x5: 00000000000000000000000000000000
x6: 00000000000000000000000000000000
x7: 00000000000000000000000000000000
x8: 00000000000000000000000000000000
x9: 00000000000000000000000000000000
x10: 00000000000000000000000000000000
x11: 00000000000000000000000000000000
x12: 00000000000000000000000000000000
x13: 0000000000000000000